# Tools 

##### Model can request to call tools that perform task such as fetching data from database, searching the web, or running code

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='The phenomenon of parrots "talking" is actually a complex mix of biological ability, social instinct, and learned behavior. It’s important to note that parrots don’t understand human language in the same way we do, but they are incredibly skilled at **mimicry**.\n\nHere are the key reasons why parrots talk:\n\n### 1. **Anatomical Adaptation: The Syrinx**\nParrots have a vocal organ called the **syrinx**, located at the base of the trachea where it branches into the lungs. This organ is unique among birds and allows them to:\n- Control airflow with two separate sides, enabling them to produce two different sounds simultaneously.\n- Vary pitch and volume with great precision.\n- Mimic a wide range of sounds, including human speech, with remarkable accuracy.\n\n### 2. **Social Instinct and Bonding**\nIn the wild, parrots are highly social flock animals. They use vocalizations to:\n- **Maintain group cohesion**: Birds call out to stay in contact with their flock.\n- **St

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get weather at a location"""
    return f"It's rainy in {location}."

model_with_tools = model.bind_tools([get_weather])


In [3]:
response = model_with_tools.invoke("What is the weather like in Kathmandu?")
print(response)
for tool_call in response.tool_calls:
    # view the tool calls made by model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': '4zfm2vcrk', 'function': {'arguments': '{"location":"Kathmandu"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 279, 'total_tokens': 308, 'completion_time': 0.081064207, 'completion_tokens_details': None, 'prompt_time': 0.020619311, 'prompt_tokens_details': None, 'queue_time': 0.051542059, 'total_time': 0.101683518}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0e085-9f74-7d71-96b5-185f9d9c9439-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Kathmandu'}, 'id': '4zfm2vcrk', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 279, 'output_tokens': 29, 'total_tokens': 308}
Tool: get_weather
Args: {'location': 'Kathmandu'}


### Tool Execution Loop

In [7]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Kathmandu?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
# print(messages)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


It's currently rainy in Kathmandu.
